In [9]:
%pip install -q google-genai
%pip install -q python-dotenv


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. A anatomia de toda chamada

Toda chamada de API de IA - em **qualquer** provedor - tem a mesma estrutura:

|Parte|O que é|No nosso código|
|-----|-------|---------------|
|**CHAVE**|Quem você é (e quem paga)|vem da ENV|
|**MODELO**|Qual IA vai atender|`gemini-2.5-flash`|
|**CONTEÚDO**|O que você quer (o prompt)|o texto recebido|

E toda resposta devolve duas coisas: o **texto gerado** e os **metadados** (incluindo os tokens consumidos).

In [10]:
import os

from dotenv import load_dotenv
from google import genai

google_api_key = os.getenv('GOOGLE_API_KEY')
client = genai.Client(api_key=(google_api_key))

print("Cliente autenticado!")

Cliente autenticado!


## 2. A requisição

Nosso conteúdo é um recibo - como o usuário do assistente digitaria

In [11]:
RECIBO = """
  Padaria Pão de Minas
  17/06/2026 - 09:22
  2x Pão de Queijo ..... R$ 9,00
  1x Café coado ........ R$ 6,50
  TOTAL ................ R$ 15,50
  Pagamento: PIX
"""

PROMPT = f"""Descreva em uma frase o que foi esta despesa, informando o valor total e a forma de pagamento

Recibo:
{RECIBO}
"""

print(PROMPT)

Descreva em uma frase o que foi esta despesa, informando o valor total e a forma de pagamento

Recibo:

  Padaria Pão de Minas
  17/06/2026 - 09:22
  2x Pão de Queijo ..... R$ 9,00
  1x Café coado ........ R$ 6,50
  TOTAL ................ R$ 15,50
  Pagamento: PIX




## 3. A primeira chamada

Repare nas três partes da anatomia: o **cliente** (autenticado com a chave), o **modelo** e o **conteúdo**

In [12]:
response = client.models.generate_content(
  model="gemini-3.5-flash", #free-tier model
  contents=PROMPT,
)

print(response.text)

Esta despesa de R$ 15,50, paga via PIX, refere-se à compra de dois pães de queijo e um café coado na Padaria Pão de Minas.


In [13]:
usage = response.usage_metadata

print("Tokens consumidos nesta chamada:")
print(f" Entrada (prompt) :      {usage.prompt_token_count}")
print(f" Saida (resposta) :      {usage.candidates_token_count}")
print(f" Racicinio :             {usage.thoughts_token_count}")
print(f" Total :                 {usage.total_token_count}")

Tokens consumidos nesta chamada:
 Entrada (prompt) :      108
 Saida (resposta) :      44
 Racicinio :             559
 Total :                 711


## 5. Tratamento básico de erros

Em produção, três erros vão aparecer:

|Erro|Código|Causa|O que fazer|
|----|------|-----|-----------|
|Autenticação|401/403|chave errada ou sem acesso|conferir .env|
|Rate limit|429|limite do tier gratuito|esperar e tentar de novo|
|Conexão|-|rede caiu, timeout|tentar de novo|


In [ ]:
from google.genai import errors

def askModel(prompt: str) -> str | None:
  """Faz uma chamada ao Gemini com tratamento dos erros mais comuns."""
  try:
    response = client.models.generate_content(
      model="gemini-3.5-flash",
      contents=prompt
    )

    return response.text
  except errors.APIError as error:
    if error.code in (401, 403):
      print("Erro de autenticação: confira sua env")
    elif error.code == 429:
      print("Rate limit: o tier gratuito tem limite por minuto. Espere ~30s e rode a célula de novo.")
    else:
      print(f"Erro da API ({error.code}): {error.message}")
    return None
  
  except Exception as error:
    print(f"Erro de conexão: {error}")
    return None


print(askModel("Em uma frase: por que guardar chave de API fora do código?"))

Para evitar o vazamento acidental de credenciais em repositórios de código e permitir a alteração de ambientes (desenvolvimento, homologação, produção) de forma segura sem precisar modificar o código-fonte.
